### 121. Best Time to Buy and Sell Stock

You are given an array `prices` where `prices[i]` is the price of a given stock on the `ith` day.

You want to maximize your profit by choosing a **single day** to buy one stock and choosing a **different day in the future** to sell that stock.

Return _the maximum profit you can achieve from this transaction_. If you cannot achieve any profit, return `0`.

```shell
Input: prices = [7, 1, 5, 3, 6, 4]
Output: 5
```
**Explanation**: Buy on day 2 (price = `1`) and sell on day 5 (price = `6`), profit = `6-1 = 5`.

Note that buying on day 2 and selling on day 1 is not allowed because you must buy before you sell.

```shell
Input: prices = [7, 6, 4, 3, 1]
Output: 0
```
**Explanation**: In this case, no transactions are done and the max profit = 0.

**Constraints**:
```shell
1 <= prices.length <= 10^5
0 <= prices[i] <= 10^4
```

---

### Reasoning

#### Step 1 - Gather Requirements
There is an **integer array** `prices` that represents stock prices of a particular day starting from day 1. By chosing one day to buy and another day to sell it I want to make the **maximum profit** that is the biggest difference between prices. When I buy the price that comes after it must be the biggest out of all left, this gives the maximum profit.

> For the buy day you pick, profit is the max price among everything after it, and you want the buy day that maximizes that difference.

- **Input**: int array
- **Output**: int
- **Sorted?**: no, it's not sorted. The array's order isn't about value at all; it represents chronological order (index 0 = day 1, index 1 = day 2, etc.). Prices can go up and down arbitrarily day to day. That's actually important: you can't assume anything monotonic about the sequence, and the order is meaningful precisely because it encodes "buy must happen before sell" — you can't reorder the array to make this easier.
- **Array length**: 1 to 100000
- **Items in the array**: 0 to 10000
- **Can the input be empty, null?** - no, assume `prices` has at **least 1 element** for this problem's base constraints — but your code should still not crash on a single-element array (you just can't make a trade, so profit is 0).
- **Duplicates**: yes, prices can repeat on consecutive or non-consecutive days (e.g. `[3, 3, 3]`)
- **Negatives**: no, prices are non-negative (you can't have a negative stock price), but profit itself can legitimately be `0` if the price only ever falls.

> One more constraint worth stating yourself: you can only make **one** transaction — buy once, sell once, and the sell day must come **after** the buy day.

#### Step 2 - Examples + Edge Cases

In [ ]:
# [7, 1, 5, 3, 6, 4] -> day 2 to buy, day 5 to cell -> 6 - 1 = 5 is maximum profit
# [1] -> 0 - one item
# [7, 1, 5, 3, 6, 4, 7, 5] -> 6 - duplicates
# [7, 7, 7, 7] -> 0 - all duplicates
# [7, 6, 4, 3, 1] -> 0 - desc ordering

- `[7, 1, 5, 3, 6, 4]` -> day 2 to buy, day 5 to cell -> 6 - 1 = 5 is maximum profit, since after buying at day 2 (1) there is only day 5 (6) suggests the biggest price to sell the stock.
- `[1]` returns 0, since there is only 1 day for buying and selling the same stock at the same price that makes it 0 (1 - 1 = 0)
- `[7, 1, 5, 3, 6, 4, 7, 5]` with duplicates, we just go ahead and check all prices after we have bougth the stock, we just trace the current difference and compare it against the one we track as maximum at this point, and update it if the current one if bigger. So, in this example we buy at day 1 and sell at day 7.
- `[7, 7, 7, 7]` -> returns 0, since buying and selling at the same price gives nothing - 0.
- `[7, 6, 4, 3, 1]` -> on desc order, we cannot make a profit, since the future prices only get smaller whenever we buy stocks. Impossible to make profit, so we return 0.

#### Step 3 Brute Force

**Nested loop - Check all possible pairs**

1. The **outer loop** traversing `len(prices) - 1` for the buying day `i`
2. Tthe **inner loop** goes through `range(i + 1, len(nums))` & looks for the selling day `j` best profit price
3. Track the `max_profit` & for each **profitable day (`j > i`)** compute the `profit`
4. Update `max_profit` with max of `profit` vs `max_profit` set previously

For `[7, 1, 5, 3, 6, 4]`:
- `7 - 1`: skip, the selling day `j` is smaller, as well as all the rest prices. Go to the next buying day `i`
- `5 - 1 = 4` gets written to `max_profit = max(j - i, max_profit)`. The rest of pairs are checked the same way

![Visual](../images/121buy_&_sell_stock_brute.png)

In [ ]:
def buy_sell_stock_brute(prices: list[int]) -> int:
    # brute force - nested loop
    # setup
    # set max price tracking: max_price = 0
    # scan the input - for loop - [0, len(prices) - 1] for buying  day
    # scan the input - for loop - [i + 1, len(prices)] for selling day

    # 1. for PROFITABLE DAY (j > i) compute the current profit: profit = j - i
    # 2. update max_profit: max(profit, max_profit)
    # 3. retrun max_profit

    max_profit = 0

    for i in range(len(prices) - 1):  # O(n - 1)
        for j in range(i + 1, len(prices)):  # O(n - i)
            if prices[j] > prices[i]:
                profit = prices[j] - prices[i]
                max_profit = max(profit, max_profit)

    return max_profit

In [ ]:
# [7, 1, 5, 3, 6, 4] -> day 2 to buy, day 5 to cell -> 6 - 1 = 5 is maximum profit
# [1] -> 0
# [7, 1, 5, 3, 6, 4, 7, 5] -> 6
# [7, 7, 7, 7] -> 0
# [7, 6, 4, 3, 1] -> 0

# buy_sell_stock_brute([7, 1, 5, 3, 6, 4])
# buy_sell_stock_brute([1])
# buy_sell_stock_brute([7, 1, 5, 3, 6, 4, 7, 5])
# buy_sell_stock_brute([7, 7, 7, 7])
buy_sell_stock_brute([7, 6, 4, 3, 1])

0

**Core Insights**:
1. Why is this $O(n^2)$?
2. What's the real, unconditional reason this is $O(n^2)$ — just in terms of what the two loops do?
3. What work is being **redundantly repeated** as `i` advances? As `i` advances from day 1 to day 2, what does the inner loop at day 2 need to know that the outer loop at day 1 already computed? What single **piece of information** could you **carry forward** instead of rescanning?

---

1. **`n-i` per iteration is still O(n²)**: iteration `i` only checks `n - i - 1` items, not `n`. Total work across all iterations is `(n-1) + (n-2) + (n-3) + ... + 1 + 0`. That's the sum of an **arithmetic sequence**, and it has a closed form: `n(n-1)/2`. Expand that: `(n² - n)/2`. When you drop constants and lower-order terms (the `-n` and the `/2`), what's left is `n²`. So even though each individual iteration does less work than the last, the **sum of a linearly-shrinking sequence** is **still quadratic** — this is worth filing permanently, you'll see this exact sum in **3Sum**, **bubble sort**, and anywhere else you have a triangular nested-loop shape.
2. **The unconditional reason:** it's structural, not data-dependent. The nested loop always performs `n(n-1)/2` comparisons for an array of size `n` — that count is fixed by the loop _shape_ alone. It doesn't matter what the values are, whether there are duplicates, or whether you add early exits that don't actually fire in the worst case. That's why "duplicates" and "no stop-early" were the wrong diagnosis: neither one is _the_ reason — the reason is **two nested loops over the same collection**, full stop.
3. For a fixed sell day `j`, the best possible profit is `prices[j] - (the smallest price among all days before j)`. In your brute force, buying day `i` and selling day `j` recomputes "what's the smallest price so far" implicitly and separately for every `i` — the **outer loop re-derives context from scratch** each time instead of **carrying it forward**.

- **`aₙ = a₁ + (n-1)d`** — gives you a **single term**, the value at position `n`. Not what we needed here.
- **`Sₙ = n/2 × (a₁ + aₙ)`** — gives you the **sum** of the first `n` terms. This is the one we needed, since we were adding up `(n-1) + (n-2) + ... + 1 + 0`.

Plug into the sum formula: `a₁ = 0` (the last, smallest term), `aₙ = n-1` (the first, biggest term), and there are `n` terms total. `Sₙ = n/2 × (0 + (n-1)) = n(n-1)/2` — that's exactly where it came from. The trick to notice: `Sₙ` is derived by pairing the first and last term, second and second-last, etc. — each pair sums to the same value `(a₁ + aₙ)`, and there are `n/2` such pairs.

#### Step 4 - Select Algorithm

**Two pointers - slow(write)/fast(read)**

1. **for loop**: `left` **writes & tracks** the **minimum price seen so far** for the buying day, `right` **reads** the prices, looks for the best profit selling price, is moved by for loop.
2. **while loop**: `left` **writes & tracks** the **minimum price seen so far**, `right` reads prices, looks for the best profit selling price.
3. `right` reaches `len(prices)`, loop ends, `max_profit = 5` — matches your Step 2 answer.
4. `left = right` jumps straight to wherever the new low actually is.

![Visual](../images/121buy_&_sell_stock2.png)

![Visual](../images/121buy_&_sell_stock.png)

> Notice: `left` never goes backward, and `prices[left]` **is** "the minimum price seen so far" — you don't need a separate `min_so_far` variable at all, `left` already points at it. That's the whole insight: one forward pass, `right` scans, `left` only jumps forward when a new low appears.

**Target time and target space complexity** for this approach, and the key insight in one sentence.

- **Time**:  $O(n)$ — single pass, `right` advances once through the array, all per-step work (comparison, `max()`, pointer move) is $O(1)$
- **Space**: $O(1)$ — only `left`, `right`, and `max_profit` tracked, no structure that grows with input

> **Key insight**: `left` always points at the **minimum price seen so far**, so no separate scan or variable is needed to track it — one forward pass compares every day against the best buy day found up to that point.

#### Step 5, 6 - Signature + Psuedocode (outline)
```py
def buy_sell_stock(prices: list[int]) -> int:
    # two pointers - slow/fast
    # setup
    # set left pointer: left = 0
    # set max profit tracking: max_profit = 0
    # set right pinter - for loop - full pass

    # 1. compare left & right
    #   right > left: compute the current profit, update max_profit with what is bigger
    #   right <= left: the min price seen so far is found -> set left at it: left = right
    # 2. right is advanced by for loop
    # 3. return max_profit
```

#### Step 7 - Write Code

**Solution 1**

In [ ]:
def buy_sell_stock(prices: list[int]) -> int:
    # two pointers - slow(write and track min price)/fast(read prices)
    # setup
    # set slow pointer: slow = 0
    # set max profit tracking: max_profit = 0
    # set fast pinter - for loop - full pass

    # 1. compare slow & fast
    #   slow > fast: compute the current profit, update max_profit with what is bigger
    #   fast <= slow: the min price seen so far is found -> set slow at it: left += 1
    # 2. fast is advanced by for loop
    # 3. return max_profit

    slow = 0
    max_profit = 0

    for fast in range(1, len(prices)):
        if prices[slow] >= prices[fast]:
            slow = fast
        else:
            profit = prices[fast] - prices[slow]
            max_profit = max(max_profit, profit)
    return max_profit

**Solution 2**.  Two pointers & `while` loop. Time $O(n)$, Space $O(1)$

In [ ]:
def buy_sell_stock2(prices: list[int]):
    slow, fast = 0, 1
    max_profit = 0

    while fast < len(prices):
        if prices[slow] >= prices[fast]:
            slow = fast
        else:
            profit = prices[fast] - prices[slow]
            max_profit = max(profit, max_profit)

        fast += 1

    return max_profit

In [3]:
buy_sell_stock2([7, 1, 5, 3, 6, 4])

5

The idea is exactly right, and it’s a clean way to think about the two-pointer solution.

- **Slow pointer:** represents the best day to buy so far—the day with the smallest price seen.
- **Fast pointer:** scans each later day as a potential selling day.
- At each fast-pointer position, calculate:
```py
profit = prices[fast] - prices[slow]
```
- If the current price is lower than the `slow` pointer’s price, move `slow` forward because you found a better buying opportunity.
- Otherwise, update the maximum profit.

A small terminology note: this is often described as a **running minimum plus one scan**, though it can also be implemented with two pointers.

For `prices = [10, 1, 5, 6, 7, 1]`:

1 1 (`price=1`) 1 — 0 2 1 (`price=1`) 5 4 4 3 1 (`price=1`) 6 5 5 4 1 (`price=1`) 7 6 6 5 5 (`price=1`) 1 — 6

> The key correctness idea is that when examining a selling day, the best valid buying day must be the **minimum price among all earlier days**. Your slow pointer maintains exactly that information.

**Complexity:**

- Time: `O(n)` — each day is processed once.
- Space: `O(1)` — only a few variables are used.

One subtle detail: update the buying pointer **before** calculating profit if the current price is lower, because buying and selling must happen on different days and the buy day must come first.

#### Step 8 - Trace the original Edge cases from Step 2

In [ ]:
buy_sell_stock([7, 1, 5, 3, 6, 4])
# buy_sell_stock([5, 1, 2, 3, 4, 0, 10])
# buy_sell_stock([7, 3, 5, 1, 3, 6, 4, 7, 5])

# buy_sell_stock([1])
# buy_sell_stock([7, 7, 7, 7])
# buy_sell_stock([7, 6, 4, 3, 1])

5

#### Step 9 - Final Time/Space complexity + further optimizations
Two pointers, slow(writes and tracks the smallest price seen so far), fast(reads prices).
- **Time**: $O(n)$, one pass with arithmetic and comparison ops of $O(1)$ time
- **Space**: $O(1)$ no extra data structure is stored along the algorithm.

**Optimization**

This solution is **already optimal** on both axes. $O(n)$ is the theoretical **floor** since every price must be examined at least once, and $O(1)$ is the **floor** for space since a numeric answer can't be produced without at least a couple of scalar variables.

### Summary
> **Pattern: same-direction two pointers with a running invariant** — `left`/slow doesn't scan independently, it just holds "the best candidate found so far" and only advances when a strictly better candidate appears. Same shape as **Move Zeros** and **Remove Duplicates**, just with the invariant being "minimum" instead of "last unique/non-zero write position." File it — you'll recognize this shape anytime brute force is "for each starting point, rescan everything after it" and the thing you're rescanning for is a running min/max.